[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Async Queries


## What you will be able to do

Query through a model: several conditions at once, the operators module for the ones comparison
cannot express, sorting, limiting and counting. Choose between `to_list`, `first_or_none` and
`get`, and say what each does when nothing matches. Run an aggregation through a model and have its
output validated into a class of your own. And recognize the two failures of that last step, which
are both about field names that do not line up.


## The idea

### The problem

`Model.find(...)` does not run anything. It builds a query, the same way a `Cursor` did, and
something has to ask for the results. Which method you use to ask decides what happens when there
are none, and the difference between them is an `IndexError` in production.

### What a find query is

An object holding a filter. `.sort()`, `.limit()`, `.skip()` and `.project()` return it again with
more on it, so a query is built up and then run once by `to_list`, `first_or_none`, `count` or
`async for`.

### Why there is an operators module

`Item.price > 10` works because Python lets a class define `>`. There is no operator for "is one of
these", so `In(Item.kind, [...])` is a function instead, and the same for `RegEx`, `And`, `Or` and
the rest. They are all in `beanie.operators` and they all build the same kind of filter.

### Where this shows up

Every read in a Beanie program. The projection model in particular is how a report gets out of the
database as a typed object rather than a pile of dictionaries, and it fails in a way that is worth
seeing once before you meet it.

### What this notebook covers

`find` with conditions, `beanie.operators`, `sort`, `limit`, `skip`. The four ways to run a query
and what each returns when nothing matches. `project` on a find, `projection_model` on an
aggregation. Then the empty-list `IndexError`, the two typos, and the projection whose names do not
match.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import asyncio

from beanie import Document, init_beanie
from pymongo import AsyncMongoClient


class Item(Document):
    sku: str
    price: float

    class Settings:
        name = "queried"


async def main():
    client = AsyncMongoClient("mongodb://127.0.0.1:27017/shop")
    await init_beanie(database=client.get_default_database(), document_models=[Item])

    await Item.delete_all()
    await Item.insert_many([Item(sku="A-1", price=10.0), Item(sku="A-2", price=20.0)])

    dear = await Item.find(Item.price > 1000).to_list()
    print("nothing matched, and that is a list:", dear)

    try:
        print(dear[0].sku)
    except IndexError as error:
        print("taking the first of it: ", type(error).__name__ + ":", error)

    print("what to write instead:    ", await Item.find(Item.price > 1000).first_or_none())
    await client.close()


asyncio.run(main())
```

```
nothing matched, and that is a list: []
taking the first of it:  IndexError: list index out of range
what to write instead:     None
```

`to_list()` gives a list, and an empty query gives an empty list, and `[0]` of an empty list is an
`IndexError` rather than anything about MongoDB. `first_or_none` is the method that exists so that
this line never has to be written.


## Setup

Eleven imports, MongoDB, the boot cell, and one helper.

- `beanie` with `Document` and `init_beanie`, and `pydantic` for the projection models
- `beanie.operators` supplies `And`, `GT`, `In`, `Or` and `RegEx`, which have no Python operator
- `pymongo` provides `AsyncMongoClient`, which is what Beanie 2 runs on, and the boot cell
  uses its synchronous `MongoClient` to check the server is up
- `subprocess`, `os`, `sys`, `time`, `random`, `version` and `PackageNotFoundError` run the boot cell

`first_problem` pulls one line out of a Pydantic `ValidationError`, which is otherwise several
paragraphs and a documentation link.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pydantic
import pymongo
from beanie import Document, init_beanie
from beanie.operators import And, GT, In, Or, RegEx
from pymongo import AsyncMongoClient

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def first_problem(error):
    """One line out of a pydantic ValidationError, which is otherwise several paragraphs."""
    problem = error.errors()[0]
    return f"{'.'.join(str(part) for part in problem['loc'])}: {problem['msg']}"


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### A model with something in it


In [2]:
class Item(Document):
    sku: str
    kind: str
    price: float
    stock: int = 0

    class Settings:
        name = "queried"


client = AsyncMongoClient(URI)
await init_beanie(database=client.get_default_database(), document_models=[Item])

await Item.delete_all()
await Item.insert_many([
    Item(sku=f"S-{number}", kind=["tool", "part"][number % 2],
         price=10.0 * number, stock=number)
    for number in range(6)
])
print("documents:", await Item.find_all().count())


documents: 6


### Conditions

Several arguments to `find` are combined with and:


In [3]:
print("tools:            ",
      [item.sku for item in await Item.find(Item.kind == "tool").to_list()])
print("tools over 10:    ",
      [item.sku for item in await Item.find(Item.kind == "tool", Item.price > 10).to_list()])
print("the same, chained:",
      [item.sku for item in await Item.find(Item.kind == "tool").find(Item.price > 10).to_list()])


tools:             ['S-0', 'S-2', 'S-4']
tools over 10:     ['S-2', 'S-4']
the same, chained: ['S-2', 'S-4']


Chaining `find` again adds a condition rather than replacing it, which is convenient for building a
query out of optional parts.

For anything Python has no operator for, the operators module:


In [4]:
print("In:      ", [item.sku for item in await Item.find(In(Item.kind, ["tool"])).to_list()])
print("Or:      ", [item.sku for item in
                    await Item.find(Or(Item.price < 15, Item.stock > 4)).to_list()])
print("And+GT:  ", [item.sku for item in
                    await Item.find(And(GT(Item.price, 10), GT(Item.stock, 2))).to_list()])
print("RegEx:   ", [item.sku for item in await Item.find(RegEx(Item.sku, "^S-[01]")).to_list()])


In:       ['S-0', 'S-2', 'S-4']
Or:       ['S-0', 'S-1', 'S-5']
And+GT:   ['S-3', 'S-4', 'S-5']
RegEx:    ['S-0', 'S-1']


`GT(Item.price, 10)` and `Item.price > 10` are the same thing written two ways. The function form
exists so that `And` and `Or` can take conditions as arguments, and because a few operators, such
as `In` and `RegEx`, have no Python equivalent at all.

### Sorting, limiting, and counting


In [5]:
print("dearest two: ",
      [item.sku for item in await Item.find_all().sort(-Item.price).limit(2).to_list()])
print("then the next:",
      [item.sku for item in await Item.find_all().sort(-Item.price).skip(2).limit(2).to_list()])
print("count:       ", await Item.find(Item.stock > 2).count())
print("all of them: ", await Item.find_all().count())


dearest two:  ['S-5', 'S-4']
then the next: ['S-3', 'S-2']
count:        3
all of them:  6


`-Item.price` is descending and `Item.price` is ascending, which is shorter than PyMongo's `-1` and
checked against the model in the same way as a filter.

### The four ways to run a query

They differ in what happens when nothing matches, which is the only part worth memorizing:


In [6]:
missing = Item.find(Item.price > 10_000)

print("to_list():       ", await missing.to_list())
print("first_or_none(): ", await missing.first_or_none())
print("count():         ", await missing.count())

found = 0
async for item in Item.find(Item.kind == "part"):                   # streams, one at a time
    found += 1
print("async for:       ", found, "documents, without building a list")


to_list():        []
first_or_none():  None
count():          0
async for:        3 documents, without building a list


`to_list()` fetches everything into memory, which is right for a page of results and wrong for a
collection. `async for` streams, which is the opposite trade and is the one to use over anything
large.

`get` is the fifth, and it takes an id rather than a filter:


In [7]:
one = await Item.find_one(Item.sku == "S-3")
print("by filter:", one.sku)
print("by id:    ", (await Item.get(one.id)).sku)
print("a missing id:", await Item.get(one.id.__class__()))


by filter: S-3
by id:     S-3
a missing id: None


### Projections

A projection model is a Pydantic class naming the fields you want. Beanie asks MongoDB for only
those and validates the result into it:


In [8]:
class JustSku(pydantic.BaseModel):
    sku: str


slim = await Item.find(Item.kind == "tool").project(JustSku).to_list()
print("type:", type(slim[0]).__name__, "| values:", [row.sku for row in slim])
print("and it has nothing else on it:", list(JustSku.model_fields))


type: JustSku | values: ['S-0', 'S-2', 'S-4']
and it has nothing else on it: ['sku']


The saving is real: a document with a large field nobody wanted is not sent at all. The gain is a
class with exactly the fields the calling code uses, which is worth more than the bytes.

### Aggregations through a model

`aggregate` takes the same pipeline as PyMongo, and `projection_model` turns each output document
into a typed object:


In [9]:
class StockByKind(pydantic.BaseModel):
    kind: str
    total: int


pipeline = [
    {"$group": {"_id": "$kind", "total": {"$sum": "$stock"}}},
    {"$project": {"_id": 0, "kind": "$_id", "total": 1}},
    {"$sort": {"kind": 1}},
]

for row in await Item.aggregate(pipeline, projection_model=StockByKind).to_list():
    print(f"  {row.kind:5} {row.total}")


  part  9
  tool  6


Note the `$project` stage renaming `_id` to `kind`. It is there because the model has a field called
`kind` and the pipeline would otherwise produce one called `_id`, and the names have to match
exactly. That is the subject of the last error below.

Without a `projection_model` you get dictionaries, which is sometimes what you want:


In [10]:
print(await Item.aggregate(pipeline).to_list())


[{'total': 9, 'kind': 'part'}, {'total': 6, 'kind': 'tool'}]


### When to reach for which

| What you want | How to write it |
|---|---|
| a filter | `Model.find(Model.field == value, ...)` |
| one of several values | `In(Model.field, [...])` |
| either condition | `Or(a, b)`, from `beanie.operators` |
| a pattern | `RegEx(Model.field, "^x")` |
| everything, in memory | `await query.to_list()` |
| one, or nothing | `await query.first_or_none()` |
| one, by id | `await Model.get(id)` |
| a large result | `async for item in query` |
| how many | `await query.count()` |
| fewer fields | `.project(SomeModel)` |
| a grouped report | `Model.aggregate(pipeline, projection_model=SomeModel)` |
| a raw filter | a dictionary, when the expressions cannot say it |

The default for "find me one" is `first_or_none`, not `to_list()[0]`. The default for a large
result is `async for`, not `to_list`.

### A search endpoint, finished


In [11]:
class ItemSummary(pydantic.BaseModel):
    sku: str
    price: float


async def search(kind=None, under=None, kinds=None, page=0, size=2):
    """Build the query out of the parts that were asked for, then run it once."""
    query = Item.find_all()
    if kind is not None:
        query = query.find(Item.kind == kind)
    if under is not None:
        query = query.find(Item.price < under)
    if kinds:
        query = query.find(In(Item.kind, kinds))

    total = await query.count()                                     # before paging
    rows = await (query.sort(Item.sku)
                       .skip(page * size)
                       .limit(size)
                       .project(ItemSummary)
                       .to_list())
    return {"total": total, "page": page, "rows": [row.model_dump() for row in rows]}


print("everything, page 0:", await search())
print("everything, page 1:", await search(page=1))
print("tools under 50:    ", await search(kind="tool", under=50))
print("either kind:       ", await search(kinds=["tool", "part"], size=3))


everything, page 0: {'total': 6, 'page': 0, 'rows': [{'sku': 'S-0', 'price': 0.0}, {'sku': 'S-1', 'price': 10.0}]}
everything, page 1: {'total': 6, 'page': 1, 'rows': [{'sku': 'S-2', 'price': 20.0}, {'sku': 'S-3', 'price': 30.0}]}
tools under 50:     {'total': 3, 'page': 0, 'rows': [{'sku': 'S-0', 'price': 0.0}, {'sku': 'S-2', 'price': 20.0}]}
either kind:        {'total': 6, 'page': 0, 'rows': [{'sku': 'S-0', 'price': 0.0}, {'sku': 'S-1', 'price': 10.0}, {'sku': 'S-2', 'price': 20.0}]}


The query is built up across several `if` statements and run twice, once for the count and once for
the page. Counting before paging is what makes `total` the number of matches rather than the number
on this page, and it is the one thing an endpoint like this gets wrong most often.

`project(ItemSummary)` means the documents crossing the network carry two fields, and
`model_dump()` turns them into dictionaries at the very edge, where JSON needs them.

### Where each part came from

| In `search` | What it relies on | The section that showed it |
|---|---|---|
| `query.find(...)` again | chaining adding a condition | Conditions |
| `In(Item.kind, kinds)` | an operator with no Python equivalent | Conditions |
| `await query.count()` | the count before paging | Sorting, limiting, and counting |
| `.sort(Item.sku)` | a sort checked against the model | Sorting, limiting, and counting |
| `.project(ItemSummary)` | fewer fields, as a typed class | Projections |
| `skip` and `limit` | paging, at the price **find and find_one** measured | **find and find_one** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/12-async-queries-solutions.ipynb).

**1.** Find the items of one kind that also cost more than a number.


In [12]:
# your code here


**2.** Use `In` to match two kinds at once.


In [13]:
# your code here


**3.** Print the two dearest items.


In [14]:
# your code here


**4.** Ask for something that matches nothing, three ways.


In [15]:
# your code here


**5.** Project a find down to one field.


In [16]:
# your code here


**6.** Run an aggregation with a projection model.


In [17]:
# your code here


## Common errors

### IndexError: list index out of range


In [18]:
nothing = await Item.find(Item.price > 10_000).to_list()
nothing[0]


IndexError: list index out of range

There is no MongoDB error here at all. The query ran, matched nothing, and `to_list()` did what it
promised: it gave a list, and the list is empty.

The pattern `(await Model.find(...).to_list())[0]` is the one to avoid, and there are two better
spellings depending on what "nothing" means to you:


In [19]:
maybe = await Item.find(Item.price > 10_000).first_or_none()
print("first_or_none:", maybe, "| and the caller decides what that means")

items = await Item.find(Item.price > 10_000).to_list()
print("or check the list:", items[0] if items else "nothing matched")


first_or_none: None | and the caller decides what that means
or check the list: nothing matched


### AttributeError: type object 'Item' has no attribute 'prce'


In [20]:
await Item.find(Item.prce < 500).to_list()


AttributeError: prce

The model knows its fields, so a typo is an error at the moment you write it rather than a query
that matches nothing. That is most of the value of using the expression form.

The same typo in a raw dictionary is a perfectly legal filter for a field nothing has:


In [21]:
quiet = await Item.find({"prce": {"$lt": 500}}).to_list()
print("as a dictionary:", len(quiet), "documents, and nothing raised")
print("the real one:   ", len(await Item.find(Item.price < 500).to_list()), "documents")


as a dictionary: 0 documents, and nothing raised
the real one:    6 documents


### pydantic_core.ValidationError: the projection model that does not match


In [22]:
class Mismatched(pydantic.BaseModel):
    kind: str
    count: int                                                      # the pipeline says "total"


try:
    await Item.aggregate(pipeline, projection_model=Mismatched).to_list()
except pydantic.ValidationError as error:
    print("ValidationError:", first_problem(error))


ValidationError: count: Field required


The pipeline produces `kind` and `total`. The model asks for `kind` and `count`. Pydantic sees a
document with no `count` in it and says the field is required, which is true and says nothing about
the real problem, which is two names that were meant to be one.

Read the pipeline's last stage and the model's fields side by side. They are a contract, written
twice:


In [23]:
print("what the pipeline produces:", sorted((await Item.aggregate(pipeline).to_list())[0]))
print("what the model wants:      ", sorted(Mismatched.model_fields))
print("what a matching model wants:", sorted(StockByKind.model_fields))


what the pipeline produces: ['kind', 'total']
what the model wants:       ['count', 'kind']
what a matching model wants: ['kind', 'total']


### No error: a projection that quietly ignores the rest


In [24]:
class OnlySku(pydantic.BaseModel):
    sku: str


rows = await Item.find_all().project(OnlySku).to_list()
print("projected:", [row.sku for row in rows][:3])

try:
    rows[0].price
except AttributeError as error:
    print("and the price is not there:", type(error).__name__ + ":", error)


projected: ['S-0', 'S-1', 'S-2']
and the price is not there: AttributeError: 'OnlySku' object has no attribute 'price'


The projection succeeded and the field is simply absent, because it was never fetched. That is the
whole point, and it becomes a problem only when the projection is added later and something
downstream still expects the full document.

By default Pydantic ignores fields it was not expecting rather than complaining, so a projection
model with a **misspelled** field is the dangerous case: the field is missing from the output and
the error, when it comes, is an `AttributeError` far away.


In [25]:
print("what a projected row actually carries:", rows[0].model_dump())
print("declared fields:", list(OnlySku.model_fields))
print("so anything downstream should take an OnlySku, not an Item")


what a projected row actually carries: {'sku': 'S-0'}
declared fields: ['sku']
so anything downstream should take an OnlySku, not an Item


In [26]:
await Item.delete_all()
await client.close()
print("tidied up and closed")


tidied up and closed


## Recap

- `Model.find(...)` builds a query and runs nothing. `to_list`, `first_or_none`, `count`, `get` and
  `async for` are the ways to run it.
- `to_list()` on no matches gives `[]`, so `[0]` is an `IndexError` with nothing to do with MongoDB.
  `first_or_none()` gives `None` and is what to write instead.
- Several arguments to `find` are combined with and, and chaining `find` again adds conditions,
  which is how an optional filter is built.
- `beanie.operators` has `In`, `Or`, `And`, `RegEx` and the rest, for the conditions Python has no
  operator for. `GT(a, b)` and `a > b` are the same thing.
- `sort(-Model.field)` descends, and every field reference is checked against the model, so a typo
  is an `AttributeError` rather than a query matching nothing.
- `async for` streams and `to_list` does not. Use the first for anything large.
- `.project(Model)` on a find and `projection_model=` on an aggregate both validate the result into
  a class of your own, and both require the field names to match what the query produced exactly.


## What is next

**Saving Changes** is writing through the model: `save` against `set` and `inc`, the state
management that `save_changes` needs turned on, and the revision that somebody else moved while you
were holding a copy.


---

&#8592; **Previous:** [Beanie Documents](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/11-beanie-documents.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Saving Changes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/13-saving-changes.ipynb) &#8594;
